## 0. Instalar las librerias y paquetes necesarios

In [1]:
!pip install --upgrade duckdb
!pip install python-calamine

In [2]:
import pandas as pd
import duckdb
import glob
import os

## 1. Cargar los datos

Se crea un archivo duckdb para persistecnia # PENDIENTE

In [3]:
con = duckdb.connect("movilidad.duckdb")   # archivo de base de datos persistente

In [4]:
RUTA_DICC = "/content/g. Diccionario base de datos.xlsx"

dom = pd.read_excel(RUTA_DICC, sheet_name="Dominios Gral", header=3).iloc[:, 1:4]
dom.columns = ["variable", "valor", "etiqueta"]
dom = dom.dropna(subset=["variable"]).astype(str)
dom["valor"] = dom["valor"].str.strip()
dom["etiqueta"] = dom["etiqueta"].str.strip()
con.execute("CREATE OR REPLACE TABLE dominios AS SELECT * FROM dom")

### 1.1. Cargar todos los módulos en DuckDB

In [5]:
SEP = ";"

for ruta in sorted(glob.glob("/content/drive/MyDrive/7mo Semestre /datos/*.csv")):
    # Reemplazamos espacios, guiones y puntos por guiones bajos para evitar errores de sintaxis en SQL
    nombre_base = os.path.splitext(os.path.basename(ruta))[0].lower()
    nombre_limpio = nombre_base.replace(" ", "_").replace("-", "_").replace(".", "_")
    nombre = f"raw_{nombre_limpio}"

    # Leemos el archivo usando Pandas para manejar de forma robusta la codificación latin1
    tmp_df = pd.read_csv(ruta, sep=SEP, encoding='latin1', dtype=str)

    # Lo registramos como una tabla real en DuckDB
    con.execute(f"CREATE OR REPLACE TABLE {nombre} AS SELECT * FROM tmp_df")

    filas = con.execute(f"SELECT COUNT(*) FROM {nombre}").fetchone()[0]
    cols  = len(con.execute(f"DESCRIBE {nombre}").fetchall())
    print(f"{nombre}: {filas:,} filas, {cols} columnas")

raw_a__modulo_a_c_x_cf: 22,755 filas, 66 columnas
raw_b__modulo_b: 67,556 filas, 26 columnas
raw_c__modulo_c1: 11,838 filas, 16 columnas
raw_d__modulo_d_e: 67,556 filas, 40 columnas
raw_e__modulo_viajes: 100,174 filas, 25 columnas
raw_f__modulo_etapas: 110,881 filas, 23 columnas


In [6]:
con.sql("SHOW TABLES").show()
con.sql("SUMMARIZE raw_e__modulo_viajes").show()

┌────────────────────────┐
│          name          │
│        varchar         │
├────────────────────────┤
│ dominios               │
│ raw_a__modulo_a_c_x_cf │
│ raw_b__modulo_b        │
│ raw_c__modulo_c1       │
│ raw_d__modulo_d_e      │
│ raw_e__modulo_viajes   │
│ raw_f__modulo_etapas   │
└────────────────────────┘

┌──────────────┬─────────────┬───────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────┬───────────────┬────────┬────────┬────────┬────────┬────────┬────────┬─────────────────┐
│ column_name  │ column_type │                                      min                                      │                                      max                                      │ approx_unique │  avg   │  std   │  q25   │  q50   │  q75   │ count  │ null_percentage │
│   varchar    │   varchar   │                                    varchar                                    │       

### 1.2. Renombrar las tablas RAW
Se establece un nuevo nombre para las tablas, con el fin de que sea mas descriptivo. La seccion 4.2 del informe se hizo con base en el siguiente codigo, junto con el Dicionario.

In [7]:
renombres = {
    "raw_a__modulo_a_c_x_cf": "raw_hogar",
    "raw_b__modulo_b":        "raw_persona",
    "raw_c__modulo_c1":       "raw_vehiculo",
    "raw_d__modulo_d_e":      "raw_persona_viajera",
    "raw_e__modulo_viajes":   "raw_viaje",
    "raw_f__modulo_etapas":   "raw_etapa",
}
for viejo, nuevo in renombres.items():
    con.execute(f"ALTER TABLE {viejo} RENAME TO {nuevo}")

con.execute("DROP VIEW IF EXISTS tmp")

Resultado despues de reenombrar las tablas

In [8]:
con.sql("SHOW TABLES").show()

┌─────────────────────┐
│        name         │
│       varchar       │
├─────────────────────┤
│ dominios            │
│ raw_etapa           │
│ raw_hogar           │
│ raw_persona         │
│ raw_persona_viajera │
│ raw_vehiculo        │
│ raw_viaje           │
└─────────────────────┘



# 2. Calidad de los Datos

## 2.1. Comprobar la conexion entre las tablas (llaves primarias)

In [9]:
con.sql("""
SELECT
  (SELECT COUNT(*) FROM raw_hogar)                                    AS hogares,
  (SELECT COUNT(DISTINCT KEY) FROM raw_hogar)                         AS hogares_key_unica,
  (SELECT COUNT(*) FROM raw_persona_viajera p
     WHERE p.PARENT_KEY IN (SELECT KEY FROM raw_hogar))               AS personas_con_hogar,
  (SELECT COUNT(*) FROM raw_persona_viajera)                          AS personas_total,
  (SELECT COUNT(*) FROM raw_viaje v
     WHERE v.PARENT_KEY IN (SELECT KEY FROM raw_persona_viajera))     AS viajes_con_persona,
  (SELECT COUNT(*) FROM raw_viaje)                                    AS viajes_total,
  (SELECT COUNT(*) FROM raw_etapa e
     WHERE e.PARENT_KEY IN (SELECT KEY FROM raw_viaje))               AS etapas_con_viaje,
  (SELECT COUNT(*) FROM raw_etapa)                                    AS etapas_total
""").show()

┌─────────┬───────────────────┬────────────────────┬────────────────┬────────────────────┬──────────────┬──────────────────┬──────────────┐
│ hogares │ hogares_key_unica │ personas_con_hogar │ personas_total │ viajes_con_persona │ viajes_total │ etapas_con_viaje │ etapas_total │
│  int64  │       int64       │       int64        │     int64      │       int64        │    int64     │      int64       │    int64     │
├─────────┼───────────────────┼────────────────────┼────────────────┼────────────────────┼──────────────┼──────────────────┼──────────────┤
│   22755 │             22755 │              67556 │          67556 │             100174 │       100174 │           110881 │       110881 │
└─────────┴───────────────────┴────────────────────┴────────────────┴────────────────────┴──────────────┴──────────────────┴──────────────┘



In [10]:
def faltantes(tabla, columnas):
    partes = [f"""SELECT '{c}' AS variable,
                  COUNT(*) AS total,
                  COUNT(*) FILTER (WHERE {c} IS NULL OR TRIM({c}) = '') AS faltantes
                  FROM {tabla}""" for c in columnas]
    q = " UNION ALL ".join(partes)
    return con.sql(f"SELECT *, ROUND(100.0*faltantes/total, 2) AS pct FROM ({q}) ORDER BY pct DESC")

faltantes("raw_hogar", ["estrato","x1","cod_loc","utam","vehicles_count","Fexp"]).show()
faltantes("raw_persona_viajera", ["Age","Fexp"]).show()
faltantes("raw_viaje", ["d11","d33","d28","stage_count"]).show()
faltantes("raw_etapa", ["d13"]).show()

# Códigos de no respuesta
con.sql("SELECT x1, COUNT(*) n FROM raw_hogar GROUP BY x1 ORDER BY n DESC").show()
con.sql("SELECT cf5, COUNT(*) n FROM raw_hogar GROUP BY cf5").show()
con.sql("SELECT d11, COUNT(*) n FROM raw_viaje GROUP BY d11 ORDER BY n DESC").show()

┌────────────────┬───────┬───────────┬────────┐
│    variable    │ total │ faltantes │  pct   │
│    varchar     │ int64 │   int64   │ double │
├────────────────┼───────┼───────────┼────────┤
│ vehicles_count │ 22755 │     13409 │  58.93 │
│ estrato        │ 22755 │      5249 │  23.07 │
│ cod_loc        │ 22755 │      5249 │  23.07 │
│ x1             │ 22755 │         0 │    0.0 │
│ utam           │ 22755 │         0 │    0.0 │
│ Fexp           │ 22755 │         0 │    0.0 │
└────────────────┴───────┴───────────┴────────┘

┌──────────┬───────┬───────────┬────────┐
│ variable │ total │ faltantes │  pct   │
│ varchar  │ int64 │   int64   │ double │
├──────────┼───────┼───────────┼────────┤
│ Fexp     │ 67556 │      2567 │    3.8 │
│ Age      │ 67556 │         0 │    0.0 │
└──────────┴───────┴───────────┴────────┘

┌─────────────┬────────┬───────────┬────────┐
│  variable   │ total  │ faltantes │  pct   │
│   varchar   │ int64  │   int64   │ double │
├─────────────┼────────┼───────────┼──

# 3. Preprocesamiento de Datos
Marco del análisis
Se definen tres cosas importantes:

- Unidad de análisis: el viaje
- Target: colectivo vs. individual motorizado, según la clasificación de d13, y qué haces con los mixtos
- Universo: viajes de residentes de Bogotá, zona urbana, encuesta completa, en modos motorizados

## 3.1. Reduccion de Dimensionalidad
Se decide que variables se van a conservar para cada una de las tablas, teniendo en cuenta que tan relevante es la variable para la pregunta problema

## 3.1.1. Raw Hogar

In [11]:
con.sql("""
SELECT cod_mpio, i4b AS municipio, COUNT(*) AS hogares
FROM raw_hogar
GROUP BY ALL
ORDER BY hogares DESC
""").show()

┌──────────┬─────────────┬─────────┐
│ cod_mpio │  municipio  │ hogares │
│ varchar  │   varchar   │  int64  │
├──────────┼─────────────┼─────────┤
│ 11001    │ Bogotá D.C. │   17506 │
│ 25754    │ Soacha      │    1442 │
│ 25473    │ Mosquera    │     381 │
│ 25214    │ Cota        │     290 │
│ 25758    │ Sopó        │     289 │
│ 25175    │ Chía        │     234 │
│ 25430    │ Madrid      │     233 │
│ 25899    │ Zipaquirá   │     214 │
│ 25099    │ Bojacá      │     212 │
│ 25269    │ Facatativá  │     205 │
│ 25785    │ Tabio       │     188 │
│ 25799    │ Tenjo       │     183 │
│ 25286    │ Funza       │     181 │
│ 25126    │ Cajicá      │     175 │
│ 25260    │ El Rosal    │     171 │
│ 25295    │ Gachancipá  │     159 │
│ 25769    │ Subachoque  │     152 │
│ 25181    │ Choachí     │     147 │
│ 25377    │ La Calera   │     139 │
│ 25740    │ Sibaté      │     130 │
│ 25817    │ Tocancipá   │     124 │
└──────────┴─────────────┴─────────┘
  21 rows                3 columns



In [12]:
con.sql("""
SELECT zona, i5a, COUNT(*) AS hogares
FROM raw_hogar
WHERE cod_mpio = '11001'
GROUP BY ALL
""").show()

┌─────────┬─────────┬─────────┐
│  zona   │   i5a   │ hogares │
│ varchar │ varchar │  int64  │
├─────────┼─────────┼─────────┤
│ 1       │ 1       │   16840 │
│ 1       │ 2       │      79 │
│ 2       │ 1       │     578 │
│ 2       │ 2       │       9 │
└─────────┴─────────┴─────────┘



In [13]:
etiquetas = {
 "c1a":"Automóvil","c1b":"Campero/Camioneta","c1c":"Pick Up/Van",
 "c1d":"Servicio especial (blanco)","c1e":"Motocicleta","c1f":"Moto-carro",
 "c1g":"Triciclo-moto","c1h":"Taxi","c1i":"Camión","c1j":"Bicitaxi sin motor",
 "c1k":"Bicitaxi eléctrico","c1l":"Bicitaxi gasolina","c1m":"Bicicleta",
 "c1n":"Bicicleta eléctrica","c1o":"Bicicleta gasolina","c1p":"Bicicleta niños",
 "c1q":"Patineta","c1r":"Patineta eléctrica","c1s":"Tracción animal",
 "c1t":"Tracción humana","c1u":"Otro"
}
cols = ", ".join(etiquetas)
con.register("etiquetas_c1",
             pd.DataFrame(list(etiquetas.items()), columns=["tipo","vehiculo"]))

con.sql(f"""
WITH largo AS (
  UNPIVOT (SELECT KEY, {cols} FROM raw_hogar
           WHERE cod_mpio = '11001' AND zona = '1')
  ON {cols}
  INTO NAME tipo VALUE valor
)
SELECT l.tipo, e.vehiculo,
       COUNT(*) FILTER (WHERE TRY_CAST(valor AS INTEGER) > 0) AS hogares_que_tienen,
       SUM(TRY_CAST(valor AS INTEGER))                        AS total_unidades,
       ROUND(100.0 * COUNT(*) FILTER (WHERE TRY_CAST(valor AS INTEGER) > 0)
             / (SELECT COUNT(*) FROM raw_hogar
                WHERE cod_mpio = '11001' AND zona = '1'), 2) AS pct_hogares
FROM largo l JOIN etiquetas_c1 e USING (tipo)
GROUP BY ALL
ORDER BY hogares_que_tienen DESC
""").show()

┌─────────┬────────────────────────────┬────────────────────┬────────────────┬─────────────┐
│  tipo   │          vehiculo          │ hogares_que_tienen │ total_unidades │ pct_hogares │
│ varchar │          varchar           │       int64        │     int128     │   double    │
├─────────┼────────────────────────────┼────────────────────┼────────────────┼─────────────┤
│ c1a     │ Automóvil                  │               4559 │           5091 │       26.95 │
│ c1m     │ Bicicleta                  │               3265 │           5166 │        19.3 │
│ c1e     │ Motocicleta                │               2672 │           2963 │       15.79 │
│ c1b     │ Campero/Camioneta          │                681 │            723 │        4.03 │
│ c1p     │ Bicicleta niños            │                231 │            289 │        1.37 │
│ c1j     │ Bicitaxi sin motor         │                122 │            200 │        0.72 │
│ c1f     │ Moto-carro                 │                 99 │         

### Módulo de hogares (`raw_hogar`)

**Distribución por municipio.** La EODH 2023 incluye 22.755 hogares de Bogotá y de 20 municipios de Bogotá Región. Bogotá D.C. (`cod_mpio = 11001`) concentra 17.506 hogares (76,9 %). Como la pregunta problema y la estrategia de DiDi se centran en Bogotá, **se conservan únicamente los hogares con residencia en Bogotá D.C.** El filtro aplica al lugar de residencia del hogar, no al destino de los viajes: se mantienen los viajes de residentes de Bogotá hacia otros municipios.

**Zona urbana/rural.** Dentro de Bogotá se compararon dos variables: `zona`, que proviene del marco muestral oficial, e `i5a`, registrada por el encuestador. Coinciden en 16.849 hogares (96,2 %) y difieren en 657 (3,8 %). La discrepancia más frecuente corresponde a 578 hogares que el marco clasifica como rurales y el encuestador registró como urbanos. Dado que DiDi opera en zona urbana, **se conservan los hogares con `zona = 1` (urbano) según el marco muestral oficial**, que deja 16.919 hogares (96,6 % de Bogotá).

**Tenencia de vehículos (`c1a`–`c1u`).** Sobre los 16.919 hogares urbanos de Bogotá, los vehículos con presencia relevante son:

| Vehículo | Hogares | % de hogares |
|---|---|---|
| Automóvil (`c1a`) | 4.559 | 26,95 % |
| Bicicleta (`c1m`) | 3.265 | 19,30 % |
| Motocicleta (`c1e`) | 2.672 | 15,79 % |
| Campero/Camioneta (`c1b`) | 681 | 4,03 % |

Los 17 tipos restantes están por debajo del 1,5 % de los hogares cada uno. Las columnas `c1a`–`c1u` registran la cantidad de unidades por tipo, y un valor vacío indica que el hogar no tiene ese vehículo (nulo estructural del cuestionario), por lo que **los vacíos se recodifican como 0** y no se tratan como datos faltantes.

PROBABLEMENTE SE VA A ELIMINAR
Para reducir la dimensionalidad y conservar la información relevante para la elección de modo, se construyen tres variables binarias:

- `tiene_auto` = 1 si el hogar tiene automóvil, campero/camioneta o pick up/van (`c1a`, `c1b`, `c1c`).
- `tiene_moto` = 1 si tiene motocicleta o moto-carro (`c1e`, `c1f`).
- `tiene_bici` = 1 si tiene bicicleta convencional o con motor (`c1m`, `c1n`, `c1o`).

Se descartan los demás tipos por su baja frecuencia (menos del 1 %) o porque corresponden a vehículos de trabajo (taxi, camión, servicio especial), que no reflejan la elección de transporte personal del hogar.

**Decisiones de limpieza derivadas** (se aplicarán en el script de limpieza, sin modificar las tablas `raw_`):
1. Filtrar `cod_mpio = '11001'`.
2. Filtrar `zona = '1'`.
3. Crear `tiene_auto`, `tiene_moto` y `tiene_bici` y descartar `c1`, `c1a`–`c1u`. (Esto se hara en la seccion de ingenieria de caracteristicas)
4. Eliminar `zat`, ya que su alta cardinalidad no aporta a la clasificación; `cod_loc` (localidad) se conserva como variable geográfica.

## Tabla de Decisiones

| # | Módulo | Variable | Decisión | Justificación | Fase en que se aplica |
|---|---|---|---|---|---|
| 1 | Hogar | cod_mpio | Filtrar = 11001 | Pregunta centrada en Bogotá | 3. Filtros |
| 2 | Hogar | cod_loc | Variable geografica | Pregunta centrada en Bogotá | 6. Ing. características |
| 3 | Hogar | zona | Filtrar = 1 | DiDi opera en zona urbana; marco oficial | 3. Filtros |
| 4 | Hogar | zat | Eliminar | Alta cardinalidad, no aporta a la pregunta | 1. Selección |
| 5 | Hogar | c1a–c1u | Conservar 8 tipos, vacío → 0 | Nulo estructural; tipos <1 % descartados | 1 y 4 |
| 6 | Hogar | tiene_auto/moto/bici | Crear | Reducir dimensionalidad de tenencia | 6. Ing. características |

In [14]:
con.execute("""
CREATE OR REPLACE TABLE sel_hogar AS
SELECT KEY, cod_mpio, zona, cod_loc, estrato, x1, cf5,
       c1a, c1b, c1c, c1e, c1f, c1m, c1n, c1o,
       fexp
FROM raw_hogar
""");

## 3.1.2. Raw Persona

In [15]:
con.sql("DESCRIBE raw_persona").show(max_rows=100)

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ OrderX      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ b4          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ b5          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ b6          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ b7          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ b8          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ b9a         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ b9b         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ b10         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ b11         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ b12         │ VARC

## Tabla de Decisiones
| Variable | Significado | Decisión | Justificación |
|---|---|---|---|
| `PARENT_KEY` | Llave del hogar | Conservar | Une con `raw_hogar` |
| `OrderX` | Número de orden de la persona en el hogar | Conservar | Une con `d2` del módulo D-E |
| `KEY` | Llave de la persona en el módulo B | Descartar | Ninguna otra tabla la usa como llave |
| `b4` | Edad en años | Solo validación | Repite `Age` (módulo D-E); se usa para comprobar consistencia y no pasa al modelo |
| `b5` | Sexo de nacimiento | Conservar | Variable básica de segmentación |
| `b6` | Autorreconocimiento étnico | Descartar | Dato sensible, no necesario para la pregunta |
| `b7` | Madre cabeza de familia | Descartar | Solo aplica a mujeres mayores de 14 años; fuera de la pregunta |
| `b8` | Máximo nivel educativo | Conservar | Indicador socioeconómico; complementa el ingreso cuando no se reporta |
| `b9a` | Actividad principal | Conservar | La ocupación condiciona el tipo de viaje y el modo |
| `b9b` | Otras actividades | Descartar | Secundaria; la actividad principal es suficiente |
| `b10` | Sector económico de la empresa | Descartar | Solo aplica a trabajadores; demasiadas categorías (21) |
| `b11` | Con quién permanece entre semana | Descartar | Solo aplica a niños de 5 a 12 años |
| `b12` | Quién lleva/recoge al niño | Descartar | Solo aplica a niños |
| `b14` | Quién cuida al niño después del colegio | Descartar | Solo aplica a niños |
| `b15` | Condiciones de discapacidad | Descartar | Dato de salud sensible y poco frecuente |
| `b16` | Inscripción en registro de discapacidad | Descartar | Dato de salud sensible |
| `b17` | Cuidador de la persona con discapacidad | Descartar | Dato de salud sensible |
| `b18` | Medios de transporte que se le dificultan | Descartar | Dato de salud sensible |
| `b19` | Licencia de conducción | Conservar | Sin licencia, el transporte individual solo es posible como pasajero (taxi, app), que es el segmento de DiDi |
| `b20` | Tenencia de celular (voz/datos) | Conservar | Pedir un servicio por app requiere datos móviles |
| `b22` | ¿Se desplazó el día anterior? | Descartar | Filtro implícito: solo quienes viajaron tienen viajes |
| `b23` | ¿Está en el hogar en este momento? | Descartar | Logística de la encuesta |
| `b24` | ¿Responde el módulo de viajes? | Descartar | Control de la encuesta |
| `b25` | ¿Responde el módulo de experiencia de viaje? | Descartar | Control de la encuesta |
| `Older18` | Indicador de mayor de edad | Descartar | Se deriva de la edad |
| `fexp` | Factor de expansión de la persona | Descartar | Los viajes usan el `fexp` del módulo D-E |

**Resumen:** de 26 variables se conservan 7 (2 llaves y 5 de análisis: sexo, educación, ocupación, licencia y celular) y `b4` se mantiene solo para validar la edad. Se descartan variables de control del formulario, redundantes, propias de subgrupos sin relación con la pregunta y datos sensibles (etnia y discapacidad), con un criterio de minimización de datos.

In [16]:
con.execute("""
CREATE OR REPLACE TABLE sel_persona AS
SELECT PARENT_KEY, OrderX, b4, b5, b8, b9a, b19, b20
FROM raw_persona
""");

## 3.1.3. Persona viajera (módulo D-E)

In [17]:
con.sql("DESCRIBE raw_persona_viajera").show(max_rows=100)

┌───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │ column_type │  null   │   key   │ default │  extra  │
│    varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ d2            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Age           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ val_b22.2     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ val_b24       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ val_b25       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ d6            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ d6_2          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ d6_3          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ d6a           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ d8_1          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NU

## Tabla de Decisiones

| Variable | Significado | Decisión | Justificación |
|---|---|---|---|
| `PARENT_KEY` | Llave del hogar | Conservar | Une con `raw_hogar` |
| `KEY` | Llave de la persona viajera | Conservar | Los viajes dependen de esta llave |
| `d2` | Número de orden de la persona | Conservar | Une con `OrderX` del módulo B |
| `Age` | Edad | Conservar | Variable de la pregunta problema |
| `d10a` | Hora de la primera salida del día | Conservar | Hora de salida del primer viaje |
| `fexp` | Factor de expansión | Conservar | Ponderación; los viajes lo heredan |
| `val_b22.2`, `val_b24`, `val_b25` | Copias de respuestas del módulo B | Descartar | Control del formulario |
| `d6a`, `journey_count` | Número de viajes declarados y registrados | Descartar | Se pueden recalcular; no explican el modo |
| `d8_1`, `d8_2`, `val_bogreg`, `zat_ref_o` | Departamento, municipio y ZAT de origen | Descartar | La ubicación ya se controla desde el hogar |
| `e1`, `e1_o`, `e2`, `e2_o`, `val_e1_e2` | Medio principal en 2019 y 2020 | Descartar | Reflejan casi el mismo target en el pasado; predicen sin explicar |
| `e3`, `e3_o`, `e4` | Razones del cambio de medio y frecuencia de viajes en pandemia | Descartar | Fuera de la pregunta problema |
| `e5`–`e7`, `e5_1`–`e7_2` | Cambios de medio por pico y placa, obras o seguridad | Descartar | Fuera de la pregunta problema |
| `e10`, `e11` | Identidad de género y orientación sexual | Descartar | Datos sensibles, no necesarios para la pregunta |
| `e16`, `e18`, `e20` | Violencia sexual sufrida, lugar y denuncia | Descartar | Datos sensibles, no necesarios para la pregunta |

**Resumen:** se conservan 6 variables (3 llaves, edad, hora de primera salida y factor de expansión). Se descartan las de control, las redundantes, las de movilidad histórica y los datos sensibles.

In [18]:
con.execute("""
CREATE OR REPLACE TABLE sel_persona_viajera AS
SELECT PARENT_KEY, KEY, d2, Age, d10a, fexp
FROM raw_persona_viajera
""");

## 3.1.4. Raw Viaje

In [19]:
con.sql("DESCRIBE raw_viaje").show(max_rows=100)

┌──────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name  │ column_type │  null   │   key   │ default │  extra  │
│   varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ OrderJourney │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ d11          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ d11_o        │ "NULL"      │ YES     │ NULL    │ NULL    │ NULL    │
│ d12          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ d12_o        │ "NULL"      │ YES     │ NULL    │ NULL    │ NULL    │
│ d12a         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ stage_count  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ d25_1        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ d25_2        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ val_bogreg2  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ cod_

## Tabla de Decisiones

| Variable | Significado | Decisión | Justificación |
|---|---|---|---|
| `PARENT_KEY` | Llave de la persona viajera | Conservar | Une con `raw_persona_viajera` |
| `KEY` | Llave del viaje | Conservar | Une con las etapas |
| `OrderJourney` | Número de orden del viaje | Conservar | Permite reconstruir la hora de salida |
| `d11` | Motivo del viaje | Conservar | Variable de la pregunta problema |
| `d28` | Hora de llegada | Conservar | Base de la duración del viaje |
| `d32` | ¿Hizo otro desplazamiento? | Conservar | Indica cuándo aplica `d33` |
| `d33` | Hora de salida hacia el siguiente viaje | Conservar | Hora de salida de los viajes después del primero |
| `d11_o`, `d12`, `d12_o` | Otro motivo y motivo de la persona acompañada | Descartar | Texto libre o subgrupo; `d11` es suficiente |
| `d12a`, `stage_count` | Número de etapas declarado y registrado | Descartar | Se recalcula desde la tabla de etapas |
| `d25_1`, `d25_2`, `val_bogreg2`, `d27`, `zat_ref_d` | Ubicación del destino | Descartar | Alta cardinalidad; fuera de la pregunta |
| `val_d28` | ¿El viaje termina el mismo día? | Descartar | Control del formulario; los cruces de medianoche se corrigen en el cálculo de la duración |
| `d29` | Frecuencia del viaje | Descartar | Fuera de la pregunta problema |
| `d30`, `d30_o`, `d31`, `d31_o` | Uso de apps para planear o guiar el viaje | Descartar | Dependen del modo elegido (consecuencia, no causa) |

**Resumen:** se conservan 7 variables: 2 llaves, orden, motivo y 3 variables de tiempo. Se descartan las redundantes, las de ubicación del destino y las que son consecuencia del modo elegido.

In [20]:
con.execute("""
CREATE OR REPLACE TABLE sel_viaje AS
SELECT PARENT_KEY, KEY, OrderJourney, d11, d28, d32, d33
FROM raw_viaje
""");

## 3.1.5. Raw Etapa

In [21]:
con.sql("DESCRIBE raw_etapa").show(max_rows=100)

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ StageX      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ d13         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ d13_o       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ d14         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ d15         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ d16         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ d17         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ d17a        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ d18         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ d18_o       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ d20         │ VARC

## Tabla de Decisiones

| Variable | Significado | Decisión | Justificación |
|---|---|---|---|
| `PARENT_KEY` | Llave del viaje | Conservar | Une con `raw_viaje` |
| `KEY` | Llave de la etapa | Conservar | Identificador único |
| `StageX` | Número de orden de la etapa | Conservar | Permite identificar el primer modo motorizado |
| `d13` | Medio de transporte de la etapa | Conservar | Base de la variable objetivo |
| `d13_o` | Otro medio (texto libre) | Descartar | Texto libre; categoría residual |
| `d14`, `d16`, `d22` | Minutos caminando y esperando | Descartar | Dependen del modo elegido (consecuencia, no causa) |
| `d15`, `d20` | Estación de abordaje y descenso | Descartar | Solo aplican a TransMilenio; alta cardinalidad |
| `d17`, `d17a`, `d18`, `d18_o` | Pago del pasaje, valor y modalidad | Descartar | Dependen del modo elegido |
| `d21a`, `d21a_o`, `Totald21a`, `d21b`, `d21b_o`, `d21h`, `d21m`, `d21c` | Vehículo usado, estacionamiento, tiempo y costo | Descartar | Solo aplican a vehículo privado; dependen del modo |
| Módulo `raw_vehiculo` completo | Detalle de cada vehículo del hogar | Descartar | La tenencia ya está resumida en hogar (`c1a`–`c1o`) |

**Resumen:** de la tabla de etapas se conservan 4 variables, y de ellas solo el modo (`d13`) es analítico. Se usa únicamente para construir la variable objetivo. El módulo de vehículos se descarta completo.

In [22]:
con.execute("""
CREATE OR REPLACE TABLE sel_etapa AS
SELECT PARENT_KEY, KEY, StageX, d13
FROM raw_etapa
""");

## Resumen

In [23]:
tablas = [
    ("hogar", "raw_hogar", "sel_hogar"),
    ("persona", "raw_persona", "sel_persona"),
    ("persona_viajera", "raw_persona_viajera", "sel_persona_viajera"),
    ("viaje", "raw_viaje", "sel_viaje"),
    ("etapa", "raw_etapa", "sel_etapa")
]

comparacion = []
for modulo, t_raw, t_sel in tablas:
    try:
        cols_raw = len(con.execute(f"DESCRIBE {t_raw}").fetchall())
    except Exception:
        cols_raw = "No encontrada"

    try:
        cols_sel = len(con.execute(f"DESCRIBE {t_sel}").fetchall())
    except Exception:
        cols_sel = "No encontrada"

    comparacion.append({
        "Módulo": modulo,
        "Variables RAW": cols_raw,
        "Variables Seleccionadas (SEL)": cols_sel
    })

df_comparacion = pd.DataFrame(comparacion)
print(df_comparacion.to_string(index=False))

         Módulo  Variables RAW  Variables Seleccionadas (SEL)
          hogar             66                             16
        persona             26                              8
persona_viajera             40                              6
          viaje             25                              7
          etapa             23                              4


**Resumen de la reduccion de dimensionalidad**  
Se pasó de [66+26+40+25+23] a [16+8+6+7] variables. De hogar se conservan ubicación, estrato, ingreso (x1), estado de la encuesta (cf5), tenencia de los vehículos relevantes y factor de expansión. De personas: edad, sexo, educación, ocupación, licencia y celular. De viajes: motivo y horas. De etapas: el modo de transporte. El módulo de vehículos se descarta, porque la tenencia ya está resumida en hogar. Se excluyen variables de control del formulario, redundantes, de subgrupos sin relación con la pregunta y datos sensibles (etnia, discapacidad, identidad de género, orientación sexual, violencia).

## 3.2. Limpieza de Datos

In [24]:
con.sql("""
SELECT 'hogar' AS tabla, COUNT(*) AS filas, COUNT(DISTINCT KEY) AS llaves_unicas,
       NULL::BIGINT AS huerfanos
FROM sel_hogar
UNION ALL
SELECT 'persona (B)', COUNT(*), COUNT(DISTINCT PARENT_KEY || '-' || CAST(OrderX AS VARCHAR)),
       COUNT(*) FILTER (WHERE PARENT_KEY NOT IN (SELECT KEY FROM sel_hogar))
FROM sel_persona
UNION ALL
SELECT 'persona viajera', COUNT(*), COUNT(DISTINCT KEY),
       COUNT(*) FILTER (WHERE PARENT_KEY NOT IN (SELECT KEY FROM sel_hogar))
FROM sel_persona_viajera
UNION ALL
SELECT 'viaje', COUNT(*), COUNT(DISTINCT KEY),
       COUNT(*) FILTER (WHERE PARENT_KEY NOT IN (SELECT KEY FROM sel_persona_viajera))
FROM sel_viaje
UNION ALL
SELECT 'etapa', COUNT(*), COUNT(DISTINCT KEY),
       COUNT(*) FILTER (WHERE PARENT_KEY NOT IN (SELECT KEY FROM sel_viaje))
FROM sel_etapa
""").show()

┌─────────────────┬────────┬───────────────┬───────────┐
│      tabla      │ filas  │ llaves_unicas │ huerfanos │
│     varchar     │ int64  │     int64     │   int64   │
├─────────────────┼────────┼───────────────┼───────────┤
│ hogar           │  22755 │         22755 │      NULL │
│ persona (B)     │  67556 │         67556 │         0 │
│ persona viajera │  67556 │         67556 │         0 │
│ viaje           │ 100174 │        100174 │         0 │
│ etapa           │ 110881 │        110881 │         0 │
└─────────────────┴────────┴───────────────┴───────────┘



Como filas = llaves_unicas no hay duplicados; huerfanos cuenta registros sin padre en la tabla superior.

In [25]:
embudo = []

def conteo(tabla):
    # Función auxiliar para contar filas en DuckDB
    return con.execute(f"SELECT COUNT(*) FROM {tabla}").fetchone()[0]

def registrar(paso):
    embudo.append({"paso": paso,
                   **{t: conteo(f"cln_{t}") for t in ["hogar", "persona_viajera", "viaje", "etapa"]}})

con.execute("""
CREATE OR REPLACE TABLE cln_hogar AS SELECT * FROM sel_hogar;
CREATE OR REPLACE TABLE cln_persona_viajera AS SELECT * FROM sel_persona_viajera;
CREATE OR REPLACE TABLE cln_viaje AS SELECT * FROM sel_viaje;
CREATE OR REPLACE TABLE cln_etapa AS SELECT * FROM sel_etapa;
""")
registrar("0. Base original")

con.execute("""
CREATE OR REPLACE TABLE cln_hogar AS
SELECT * FROM sel_hogar
WHERE CAST(cod_mpio AS INTEGER) = 11001
  AND CAST(zona AS INTEGER) = 1
  AND CAST(cf5 AS INTEGER) = 1;

CREATE OR REPLACE TABLE cln_persona_viajera AS
SELECT * FROM sel_persona_viajera WHERE PARENT_KEY IN (SELECT KEY FROM cln_hogar);

CREATE OR REPLACE TABLE cln_viaje AS
SELECT * FROM sel_viaje WHERE PARENT_KEY IN (SELECT KEY FROM cln_persona_viajera);

CREATE OR REPLACE TABLE cln_etapa AS
SELECT * FROM sel_etapa WHERE PARENT_KEY IN (SELECT KEY FROM cln_viaje);
""")
registrar("1. Bogotá, urbano, encuesta completa")
pd.DataFrame(embudo)

,paso,hogar,persona_viajera,viaje,etapa
0,0. Base original,22755,67556,100174,110881
1,"1. Bogotá, urbano, encuesta completa",16919,48988,74758,82604


Se copian las tablas sel_ a cln_ y se registra el conteo inicial: es la fila 0 del embudo.
1. Se filtran los hogares con tres condiciones: Bogotá (cod_mpio = 11001), zona urbana (zona = 1) y encuesta completa (cf5 = 1).
2. El filtro baja en cascada por las llaves: solo quedan las personas de esos hogares, los viajes de esas personas y las etapas de esos viajes.

Se registra la fila 1 del embudo.

In [26]:
con.execute("""
CREATE OR REPLACE TABLE map_modo AS
SELECT CAST(valor AS INTEGER) AS cod_modo,
       etiqueta   AS modo,
       CASE WHEN CAST(valor AS INTEGER) IN (1, 2, 3, 4, 5, 6, 7, 10, 13)                  THEN 'colectivo'
            WHEN CAST(valor AS INTEGER) IN (8, 11, 12, 14, 20, 21, 22, 23, 24, 25, 30)    THEN 'individual_motorizado'
            ELSE 'excluido' END AS tipo_modo
FROM dominios WHERE variable = 'd13';

CREATE OR REPLACE TABLE viaje_modo AS
SELECT e.PARENT_KEY AS key_viaje,
       COUNT(*) AS num_etapas,
       COUNT(*) FILTER (WHERE m.cod_modo IS NULL) AS etapas_sin_modo,
       BOOL_OR(m.tipo_modo = 'colectivo')              AS usa_colectivo,
       BOOL_OR(m.tipo_modo = 'individual_motorizado')  AS usa_individual,
       arg_min(m.cod_modo, TRY_CAST(e.StageX AS INTEGER)) FILTER (WHERE m.tipo_modo = 'colectivo')             AS modo_col,
       arg_min(m.cod_modo, TRY_CAST(e.StageX AS INTEGER)) FILTER (WHERE m.tipo_modo = 'individual_motorizado') AS modo_ind
FROM cln_etapa e
LEFT JOIN map_modo m ON TRY_CAST(e.d13 AS INTEGER) = m.cod_modo
GROUP BY e.PARENT_KEY;

CREATE OR REPLACE TABLE viaje_modo AS
SELECT *,
       CASE WHEN usa_colectivo AND NOT usa_individual THEN 'colectivo'
            WHEN usa_individual AND NOT usa_colectivo THEN 'individual_motorizado'
            WHEN usa_colectivo AND usa_individual     THEN 'mixto'
            ELSE 'no_motorizado' END AS tipo_transporte,
       CASE WHEN usa_colectivo AND NOT usa_individual THEN modo_col
            WHEN usa_individual AND NOT usa_colectivo THEN modo_ind END AS cod_modo_principal
FROM viaje_modo;
""")

con.sql("""
SELECT tipo_transporte, COUNT(*) AS viajes,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM viaje_modo GROUP BY ALL ORDER BY viajes DESC
""").show()
con.sql("SELECT SUM(etapas_sin_modo) AS etapas_sin_modo_valido FROM viaje_modo").show()

┌───────────────────────┬────────┬────────┐
│    tipo_transporte    │ viajes │  pct   │
│        varchar        │ int64  │ double │
├───────────────────────┼────────┼────────┤
│ no_motorizado         │  33158 │  44.35 │
│ colectivo             │  23583 │  31.55 │
│ individual_motorizado │  17830 │  23.85 │
│ mixto                 │    187 │   0.25 │
└───────────────────────┴────────┴────────┘

┌────────────────────────┐
│ etapas_sin_modo_valido │
│         int128         │
├────────────────────────┤
│                      0 │
└────────────────────────┘



Cada viaje se clasifica según sus etapas: colectivo si solo usa modos colectivos, individual motorizado si solo usa modos individuales motorizados, mixto si combina ambos y no motorizado si no usa ninguno (a pie, bicicleta, etc.). Se excluyen los mixtos y los no motorizados. El modo principal es el primer modo motorizado del viaje.

In [27]:
con.execute("""
CREATE OR REPLACE TABLE cln_viaje AS
SELECT v.*, vm.tipo_transporte, vm.cod_modo_principal, vm.num_etapas
FROM cln_viaje v
JOIN viaje_modo vm ON v.KEY = vm.key_viaje
WHERE vm.tipo_transporte IN ('colectivo', 'individual_motorizado');

CREATE OR REPLACE TABLE cln_etapa AS
SELECT * FROM cln_etapa WHERE PARENT_KEY IN (SELECT KEY FROM cln_viaje);

CREATE OR REPLACE TABLE cln_persona_viajera AS
SELECT * FROM cln_persona_viajera WHERE KEY IN (SELECT PARENT_KEY FROM cln_viaje);

CREATE OR REPLACE TABLE cln_hogar AS
SELECT * FROM cln_hogar WHERE KEY IN (SELECT PARENT_KEY FROM cln_persona_viajera);
""")
registrar("2. Solo viajes colectivos o individuales motorizados")
pd.DataFrame(embudo)

,paso,hogar,persona_viajera,viaje,etapa
0,0. Base original,22755,67556,100174,110881
1,"1. Bogotá, urbano, encuesta completa",16919,48988,74758,82604
2,2. Solo viajes colectivos o individuales motor...,12096,20925,41413,48837


In [28]:
con.execute("""
CREATE OR REPLACE TABLE cln_persona_unida AS
SELECT v.PARENT_KEY, v.KEY, v.d2, v.Age, v.d10a, v.fexp,
       p.b4, p.b5, p.b8, p.b9a, p.b19, p.b20
FROM cln_persona_viajera v
LEFT JOIN sel_persona p
       ON p.PARENT_KEY = v.PARENT_KEY AND CAST(p.OrderX AS INTEGER) = CAST(v.d2 AS INTEGER)
""")

con.sql("""
SELECT COUNT(*) AS personas,
       COUNT(*) FILTER (WHERE b5 IS NULL) AS sin_pareja_modulo_b,
       COUNT(*) FILTER (WHERE TRY_CAST(b4 AS INTEGER) = TRY_CAST(Age AS INTEGER)) AS edad_coincide,
       COUNT(*) FILTER (WHERE TRY_CAST(b4 AS INTEGER) <> TRY_CAST(Age AS INTEGER)) AS edad_difiere
FROM cln_persona_unida
""").show()

┌──────────┬─────────────────────┬───────────────┬──────────────┐
│ personas │ sin_pareja_modulo_b │ edad_coincide │ edad_difiere │
│  int64   │        int64        │     int64     │    int64     │
├──────────┼─────────────────────┼───────────────┼──────────────┤
│    20925 │                   0 │         20923 │            2 │
└──────────┴─────────────────────┴───────────────┴──────────────┘



Como hay dos registros donde la edad difiere, se eliminan

In [29]:
con.execute("""
CREATE OR REPLACE TABLE cln_persona_unida AS
SELECT *
FROM cln_persona_unida
WHERE TRY_CAST(b4 AS INTEGER) = TRY_CAST(Age AS INTEGER)
""")

con.sql("""
SELECT COUNT(*) AS personas_restantes,
       COUNT(*) FILTER (WHERE TRY_CAST(b4 AS INTEGER) <> TRY_CAST(Age AS INTEGER)) AS edad_difiere
FROM cln_persona_unida
""").show()

┌────────────────────┬──────────────┐
│ personas_restantes │ edad_difiere │
│       int64        │    int64     │
├────────────────────┼──────────────┤
│              20923 │            0 │
└────────────────────┴──────────────┘



Valores faltantes

In [30]:
try:
    con.rollback()
except Exception:
    pass

def faltantes(tabla, reglas):
    q = " UNION ALL ".join(
        f"SELECT '{v}' AS variable, COUNT(*) AS total, "
        f"COUNT(*) FILTER (WHERE {cond}) AS faltantes FROM {tabla}"
        for v, cond in reglas.items())
    return con.sql(f"SELECT *, ROUND(100.0 * faltantes / total, 2) AS pct FROM ({q}) ORDER BY pct DESC").df()

print("--- FALTANTES HOGAR ---")
display(faltantes("cln_hogar", {
    "estrato (fuera de 1-6)":   "TRY_CAST(estrato AS INTEGER) IS NULL OR TRY_CAST(estrato AS INTEGER) NOT BETWEEN 1 AND 6",
    "x1 ingreso (vacío o NS/NR)": "TRY_CAST(x1 AS INTEGER) IS NULL OR TRY_CAST(x1 AS INTEGER) = 12",
    "cod_loc":                  "TRY_CAST(cod_loc AS INTEGER) IS NULL",
    "fexp": "TRY_CAST(REPLACE(TRIM(fexp), ',', '.') AS DOUBLE) IS NULL OR TRY_CAST(REPLACE(TRIM(fexp), ',', '.') AS DOUBLE) <= 0",
}))

print("\n--- FALTANTES PERSONA UNIDA ---")
display(faltantes("cln_persona_unida", {
    "Age (vacía o fuera de 5-100)": "TRY_CAST(Age AS INTEGER) IS NULL OR TRY_CAST(Age AS INTEGER) NOT BETWEEN 5 AND 100",
    "b5 sexo":                  "TRY_CAST(b5 AS INTEGER) IS NULL",
    "b8 educación":             "TRY_CAST(b8 AS INTEGER) IS NULL",
    "b9a actividad":            "TRY_CAST(b9a AS INTEGER) IS NULL",
    "b19 licencia (16+ años)":  "TRY_CAST(Age AS INTEGER) >= 16 AND (b19 IS NULL OR TRIM(CAST(b19 AS VARCHAR)) = '')",
    "b19 licencia (<16, estructural)": "TRY_CAST(Age AS INTEGER) < 16 AND (b19 IS NULL OR TRIM(CAST(b19 AS VARCHAR)) = '')",
    "b20 celular (vacío o NS/NR)": "TRY_CAST(b20 AS INTEGER) IS NULL OR TRY_CAST(b20 AS INTEGER) = 5",
    "fexp": "TRY_CAST(REPLACE(TRIM(fexp), ',', '.') AS DOUBLE) IS NULL OR TRY_CAST(REPLACE(TRIM(fexp), ',', '.') AS DOUBLE) <= 0",
}))

print("\n--- FALTANTES VIAJE ---")
display(faltantes("cln_viaje", {
    "d11 motivo (vacío o NS/NR)": "TRY_CAST(d11 AS INTEGER) IS NULL OR TRY_CAST(d11 AS INTEGER) = 17",
    "d28 hora llegada":         "TRY_CAST(d28 AS TIME) IS NULL",
}))

--- FALTANTES HOGAR ---


,variable,total,faltantes,pct
0,x1 ingreso (vacío o NS/NR),12096,2657,21.97
1,cod_loc,12096,0,0.00
2,estrato (fuera de 1-6),12096,0,0.00
3,fexp,12096,0,0.00



--- FALTANTES PERSONA UNIDA ---


,variable,total,faltantes,pct
0,"b19 licencia (<16, estructural)",20923,765,3.66
1,b20 celular (vacío o NS/NR),20923,453,2.17
2,b5 sexo,20923,0,0.00
3,Age (vacía o fuera de 5-100),20923,0,0.00
4,b9a actividad,20923,0,0.00
5,b8 educación,20923,0,0.00
6,b19 licencia (16+ años),20923,1,0.00
7,fexp,20923,0,0.00



--- FALTANTES VIAJE ---


,variable,total,faltantes,pct
0,d11 motivo (vacío o NS/NR),41413,7,0.02
1,d28 hora llegada,41413,0,0.00


In [31]:
con.execute("""
-- 1. Reiniciar y filtrar hogares de forma segura con estrato y fexp corregido
CREATE OR REPLACE TABLE cln_hogar AS
SELECT * FROM sel_hogar
WHERE CAST(cod_mpio AS INTEGER) = 11001
  AND CAST(zona AS INTEGER) = 1
  AND CAST(cf5 AS INTEGER) = 1
  AND TRY_CAST(estrato AS INTEGER) BETWEEN 1 AND 6
  AND TRY_CAST(REPLACE(fexp, ',', '.') AS DOUBLE) > 0;

-- 2. Volver a unir y filtrar personas usando la relación sociodemográfica
CREATE OR REPLACE TABLE cln_persona_unida AS
SELECT v.PARENT_KEY, v.KEY, v.d2, v.Age, v.d10a, v.fexp,
       p.b4, p.b5, p.b8, p.b9a, p.b19, p.b20
FROM sel_persona_viajera v
LEFT JOIN sel_persona p
       ON p.PARENT_KEY = v.PARENT_KEY AND CAST(p.OrderX AS INTEGER) = CAST(v.d2 AS INTEGER)
WHERE v.PARENT_KEY IN (SELECT KEY FROM cln_hogar)
  AND TRY_CAST(v.Age AS INTEGER) BETWEEN 5 AND 100
  AND TRY_CAST(p.b5 AS INTEGER) IS NOT NULL
  AND TRY_CAST(p.b8 AS INTEGER) IS NOT NULL
  AND TRY_CAST(p.b9a AS INTEGER) IS NOT NULL
  AND TRY_CAST(p.b20 AS INTEGER) BETWEEN 1 AND 4
  AND TRY_CAST(REPLACE(v.fexp, ',', '.') AS DOUBLE) > 0;

-- 3. Filtrar viajes basados en las personas válidas y motivos
CREATE OR REPLACE TABLE cln_viaje AS
SELECT v.*, vm.tipo_transporte, vm.cod_modo_principal, vm.num_etapas
FROM sel_viaje v
JOIN viaje_modo vm ON v.KEY = vm.key_viaje
WHERE v.PARENT_KEY IN (SELECT KEY FROM cln_persona_unida)
  AND vm.tipo_transporte IN ('colectivo', 'individual_motorizado')
  AND TRY_CAST(v.d11 AS INTEGER) BETWEEN 1 AND 16;

-- 4. Mantener cascada de consistencia referencial en el resto de tablas
CREATE OR REPLACE TABLE cln_etapa AS
SELECT * FROM sel_etapa WHERE PARENT_KEY IN (SELECT KEY FROM cln_viaje);

CREATE OR REPLACE TABLE cln_persona_unida AS
SELECT * FROM cln_persona_unida WHERE KEY IN (SELECT PARENT_KEY FROM cln_viaje);

CREATE OR REPLACE TABLE cln_persona_viajera AS
SELECT * FROM sel_persona_viajera WHERE KEY IN (SELECT KEY FROM cln_persona_unida);

CREATE OR REPLACE TABLE cln_hogar AS
SELECT * FROM cln_hogar WHERE KEY IN (SELECT PARENT_KEY FROM cln_persona_unida);
""")

embudo = [
    {"paso": "0. Base original", "hogar": 22755, "persona_viajera": 67556, "viaje": 100174, "etapa": 110881},
    {"paso": "1. Bogotá, urbano, encuesta completa", "hogar": 16919, "persona_viajera": 48988, "viaje": 74758, "etapa": 82604},
    {"paso": "2. Solo viajes colectivos o individuales motorizados", "hogar": 12096, "persona_viajera": 20925, "viaje": 41413, "etapa": 48837}
]
registrar("3. Sin faltantes en variables clave")
pd.DataFrame(embudo)

,paso,hogar,persona_viajera,viaje,etapa
0,0. Base original,22755,67556,100174,110881
1,"1. Bogotá, urbano, encuesta completa",16919,48988,74758,82604
2,2. Solo viajes colectivos o individuales motor...,12096,20925,41413,48837
3,3. Sin faltantes en variables clave,11917,20471,40529,47807


#### Decisiones – Limpieza

| Paso | Decisión | Justificación |
|---|---|---|
| Llaves | Verificar unicidad y registros huérfanos antes de filtrar | Evitar duplicados en los JOINs |
| Universo geográfico | Conservar hogares con `cod_mpio = 11001` y `zona = 1` | La pregunta y la estrategia de DiDi son para Bogotá urbana |
| Calidad de la encuesta | Conservar hogares con `cf5 = 1` (completa) | Las encuestas incompletas o rechazadas no tienen información confiable |
| Cascada de filtros | Aplicar los filtros de hogar → persona → viaje → etapa | Mantener la integridad referencial |
| Variable objetivo | Clasificar cada viaje como colectivo, individual motorizado, mixto o no motorizado según sus etapas | El modo se registra por etapa, no por viaje |
| Viajes mixtos y no motorizados | Excluir | No representan la elección colectivo vs. individual |
| Buses escolares y de empresa (d13 = 15, 16) | Excluir | La persona no elige ese modo |
| Unión módulos B y D-E | `PARENT_KEY` + `OrderX` = `PARENT_KEY` + `d2` | Integrar características sociodemográficas con la persona viajera |
| Estrato fuera de 1–6 | Eliminar el registro | Valor inválido; porcentaje bajo [__ %] |
| Edad vacía o fuera de 5–100 | Eliminar el registro | Valor inválido o atípico |
| Sexo, educación, actividad vacíos | Eliminar el registro | Porcentaje bajo [__ %] |
| Celular NS/NR (`b20 = 5`) | Eliminar el registro | Porcentaje bajo [__ %] |
| Motivo NS/NR (`d11 = 17`) | Eliminar el viaje | No permite caracterizar el viaje |
| Ingreso NS/NR (`x1 = 12`) | Conservar como categoría "No reporta" | Porcentaje alto [__ %]; eliminarlo sesgaría la muestra |
| Licencia vacía en menores de 16 | Conservar | Nulo estructural: la pregunta no aplica |
| Factor de expansión vacío o ≤ 0 | Eliminar el registro | Necesario para la ponderación |

**Resumen de limpieza.**
- Las llaves son únicas y no hay registros huérfanos
- Universo: hogares de Bogotá (`cod_mpio = 11001`), zona urbana (`zona = 1`) y encuesta completa (`cf5 = 1`), con viajes 100 % colectivos o 100 % individuales motorizados. Se excluyeron [__] viajes mixtos y [__] no motorizados.
- Se eliminaron los registros con faltantes en estrato, edad, sexo, educación, actividad, celular, motivo y factor de expansión, que representaban menos del [__] % en cada variable.
- **Ingreso (`x1`)**: el [__] % responde NS/NR. No se elimina; se conserva como categoría "No reporta".
- **Licencia (`b19`)**: los vacíos en menores de 16 años son estructurales (la pregunta no aplica).
- El embudo final queda en [__] hogares, [__] personas y [__] viajes.

Ingenieria de caracteristicas

In [33]:
con.execute("""
CREATE OR REPLACE TABLE fe_hogar AS
SELECT h.KEY AS key_hogar,
       TRY_CAST(h.cod_loc AS INTEGER) AS cod_localidad,
       l.etiqueta     AS localidad,
       TRY_CAST(h.estrato AS INTEGER) AS estrato,
       CASE WHEN TRY_CAST(h.estrato AS INTEGER) <= 2 THEN 'Bajo (1-2)'
            WHEN TRY_CAST(h.estrato AS INTEGER) <= 4 THEN 'Medio (3-4)'
            ELSE 'Alto (5-6)' END AS estrato_grupo,
       TRY_CAST(h.x1 AS INTEGER) AS cod_ingreso,
       CASE WHEN TRY_CAST(h.x1 AS INTEGER) BETWEEN 1 AND 3  THEN '1. Hasta $1,16 M'
            WHEN TRY_CAST(h.x1 AS INTEGER) BETWEEN 4 AND 6  THEN '2. $1,16 M - $2,5 M'
            WHEN TRY_CAST(h.x1 AS INTEGER) BETWEEN 7 AND 8  THEN '3. $2,5 M - $4,9 M'
            WHEN TRY_CAST(h.x1 AS INTEGER) BETWEEN 9 AND 11 THEN '4. Más de $4,9 M'
            ELSE '5. No reporta' END AS ingreso_grupo,
       CAST(COALESCE(TRY_CAST(c1a AS INTEGER), 0) + COALESCE(TRY_CAST(c1b AS INTEGER), 0) + COALESCE(TRY_CAST(c1c AS INTEGER), 0) > 0 AS INTEGER) AS tiene_auto,
       CAST(COALESCE(TRY_CAST(c1e AS INTEGER), 0) + COALESCE(TRY_CAST(c1f AS INTEGER), 0) > 0 AS INTEGER) AS tiene_moto,
       CAST(COALESCE(TRY_CAST(c1m AS INTEGER), 0) + COALESCE(TRY_CAST(c1n AS INTEGER), 0) + COALESCE(TRY_CAST(c1o AS INTEGER), 0) > 0 AS INTEGER) AS tiene_bici
FROM cln_hogar h
LEFT JOIN dominios l ON l.variable = 'cod_i4c' AND TRY_CAST(l.valor AS INTEGER) = TRY_CAST(h.cod_loc AS INTEGER)
""")
con.sql("SELECT * FROM fe_hogar LIMIT 5").show()

┌───────────────────────────────────────────┬───────────────┬────────────────┬─────────┬───────────────┬─────────────┬─────────────────────┬────────────┬────────────┬────────────┐
│                 key_hogar                 │ cod_localidad │   localidad    │ estrato │ estrato_grupo │ cod_ingreso │    ingreso_grupo    │ tiene_auto │ tiene_moto │ tiene_bici │
│                  varchar                  │     int32     │    varchar     │  int32  │    varchar    │    int32    │       varchar       │   int32    │   int32    │   int32    │
├───────────────────────────────────────────┼───────────────┼────────────────┼─────────┼───────────────┼─────────────┼─────────────────────┼────────────┼────────────┼────────────┤
│ uuid:e3b750ec-55d0-4b6b-8b58-36eac88be4eb │            15 │ Antonio Nariño │       2 │ Bajo (1-2)    │          12 │ 5. No reporta       │          0 │          0 │          0 │
│ uuid:91ff0f20-dab5-4a43-8668-154399f789b0 │            15 │ Antonio Nariño │       2 │ Bajo (1-2) 

In [35]:
con.execute("""
CREATE OR REPLACE TABLE fe_persona AS
SELECT KEY AS key_persona,
       PARENT_KEY AS key_hogar,
       TRY_CAST(Age AS INTEGER) AS edad,
       CASE WHEN TRY_CAST(Age AS INTEGER) < 18 THEN '05-17'
            WHEN TRY_CAST(Age AS INTEGER) <= 25 THEN '18-25'
            WHEN TRY_CAST(Age AS INTEGER) <= 35 THEN '26-35'
            WHEN TRY_CAST(Age AS INTEGER) <= 45 THEN '36-45'
            WHEN TRY_CAST(Age AS INTEGER) <= 60 THEN '46-60'
            ELSE '61+' END AS rango_edad,
       CASE TRY_CAST(b5 AS INTEGER) WHEN 1 THEN 'Hombre' WHEN 2 THEN 'Mujer' ELSE 'Intersexual' END AS sexo,
       CASE WHEN TRY_CAST(b8 AS INTEGER) IN (1, 2, 3, 14) THEN '1. Ninguno o primaria'
            WHEN TRY_CAST(b8 AS INTEGER) BETWEEN 4 AND 7  THEN '2. Secundaria o media'
            WHEN TRY_CAST(b8 AS INTEGER) IN (8, 9)        THEN '3. Técnico o tecnológico'
            WHEN TRY_CAST(b8 AS INTEGER) IN (10, 11)      THEN '4. Universitario'
            ELSE '5. Posgrado' END AS nivel_educativo,
       CASE WHEN TRY_CAST(b9a AS INTEGER) BETWEEN 1 AND 5                 THEN 'Estudiante'
            WHEN TRY_CAST(b9a AS INTEGER) IN (11, 13, 14, 16, 17)         THEN 'Empleado'
            WHEN TRY_CAST(b9a AS INTEGER) IN (12, 18, 19, 20, 21)         THEN 'Independiente o informal'
            WHEN TRY_CAST(b9a AS INTEGER) = 31                            THEN 'Hogar'
            WHEN TRY_CAST(b9a AS INTEGER) IN (32, 36)                     THEN 'Pensionado o rentista'
            ELSE 'Sin ocupación u otra' END AS ocupacion,
       CAST(b19 LIKE '%1%' OR b19 LIKE '%3%' AS INTEGER) AS licencia_auto,
       CAST(b19 LIKE '%2%' AS INTEGER)                    AS licencia_moto,
       CAST(TRY_CAST(b20 AS INTEGER) IN (3, 4) AS INTEGER)                 AS celular_datos,
       TRY_CAST(REPLACE(fexp, ',', '.') AS DOUBLE) AS fexp
FROM cln_persona_unida
""")
con.sql("SELECT * FROM fe_persona LIMIT 5").show()

┌──────────────────────────────────────────────────────────────────┬───────────────────────────────────────────┬───────┬────────────┬─────────┬───────────────────────┬──────────────────────────┬───────────────┬───────────────┬───────────────┬─────────────┐
│                           key_persona                            │                 key_hogar                 │ edad  │ rango_edad │  sexo   │    nivel_educativo    │        ocupacion         │ licencia_auto │ licencia_moto │ celular_datos │    fexp     │
│                             varchar                              │                  varchar                  │ int32 │  varchar   │ varchar │        varchar        │         varchar          │     int32     │     int32     │     int32     │   double    │
├──────────────────────────────────────────────────────────────────┼───────────────────────────────────────────┼───────┼────────────┼─────────┼───────────────────────┼──────────────────────────┼───────────────┼───────────────┼───

In [37]:
con.execute("""
-- Creamos una función SQL macro en DuckDB para convertir el formato de hora de la encuesta (ej. '05:30:00 a. m.' o '05:30:00 p. m.') a minutos totales del día
CREATE OR REPLACE TEMP MACRO convertir_a_minutos(hora_str) AS
  CASE
    WHEN hora_str IS NULL OR TRIM(hora_str) = '' THEN NULL
    ELSE
      (TRY_CAST(SPLIT_PART(hora_str, ':', 1) AS INTEGER) % 12 +
       CASE WHEN LOWER(hora_str) LIKE '%p.%m%' OR LOWER(hora_str) LIKE '%pm%' THEN 12 ELSE 0 END) * 60 +
      TRY_CAST(SPLIT_PART(hora_str, ':', 2) AS INTEGER)
  END;
""")

con.execute("""
CREATE OR REPLACE TABLE hora_viaje AS
SELECT v.KEY AS key_viaje,
       CASE WHEN TRY_CAST(v.OrderJourney AS INTEGER) = 1 THEN convertir_a_minutos(p.d10a)
            ELSE LAG(convertir_a_minutos(v.d33)) OVER (PARTITION BY v.PARENT_KEY ORDER BY TRY_CAST(v.OrderJourney AS INTEGER))
       END AS salida_min,
       convertir_a_minutos(v.d28) AS llegada_min
FROM sel_viaje v
LEFT JOIN sel_persona_viajera p ON v.PARENT_KEY = p.KEY;

CREATE OR REPLACE TABLE fe_viaje AS
SELECT v.KEY AS key_viaje,
       v.PARENT_KEY AS key_persona,
       TRY_CAST(v.OrderJourney AS INTEGER) AS orden_viaje,
       TRY_CAST(v.d11 AS INTEGER) AS cod_motivo,
       v.cod_modo_principal,
       v.num_etapas,
       h.salida_min,
       CASE WHEN h.salida_min IS NULL      THEN 'Sin dato'
            WHEN h.salida_min < 300        THEN '1. Madrugada (0-5 h)'
            WHEN h.salida_min < 540        THEN '2. Pico AM (5-9 h)'
            WHEN h.salida_min < 960        THEN '3. Valle (9-16 h)'
            WHEN h.salida_min < 1200       THEN '4. Pico PM (16-20 h)'
            ELSE '5. Noche (20-24 h)' END AS franja,
       CASE WHEN dur BETWEEN 1 AND 300 THEN dur END AS duracion_min,
       v.tipo_transporte,
       CAST(v.tipo_transporte = 'individual_motorizado' AS INTEGER) AS es_individual
FROM cln_viaje v
LEFT JOIN (
    SELECT *,
           CASE WHEN llegada_min - salida_min < 0 THEN llegada_min - salida_min + 1440
                ELSE llegada_min - salida_min END AS dur
    FROM hora_viaje
) h ON h.key_viaje = v.KEY
""")

con.sql("""
SELECT COUNT(*) AS viajes,
       COUNT(*) FILTER (WHERE salida_min IS NULL)   AS sin_hora_salida,
       COUNT(*) FILTER (WHERE duracion_min IS NULL) AS sin_duracion_valida,
       MEDIAN(duracion_min) AS duracion_mediana
FROM fe_viaje
""").show()

┌────────┬─────────────────┬─────────────────────┬──────────────────┐
│ viajes │ sin_hora_salida │ sin_duracion_valida │ duracion_mediana │
│ int64  │      int64      │        int64        │      double      │
├────────┼─────────────────┼─────────────────────┼──────────────────┤
│  40529 │               0 │                  10 │             60.0 │
└────────┴─────────────────┴─────────────────────┴──────────────────┘



In [38]:
con.execute("""
CREATE OR REPLACE TABLE map_motivo AS
SELECT TRY_CAST(valor AS INTEGER) AS cod_motivo,
       etiqueta   AS motivo,
       CASE WHEN TRY_CAST(valor AS INTEGER) IN (1, 15)          THEN 'Trabajo'
            WHEN TRY_CAST(valor AS INTEGER) = 2                 THEN 'Estudio'
            WHEN TRY_CAST(valor AS INTEGER) = 5                 THEN 'Regreso al hogar'
            WHEN TRY_CAST(valor AS INTEGER) IN (3, 6)           THEN 'Salud o trámites'
            WHEN TRY_CAST(valor AS INTEGER) = 8                 THEN 'Compras'
            WHEN TRY_CAST(valor AS INTEGER) IN (4, 9, 10, 12)   THEN 'Ocio, social o religioso'
            WHEN TRY_CAST(valor AS INTEGER) IN (7, 13, 14)      THEN 'Acompañar o llevar'
            ELSE 'Otro' END AS motivo_grupo
FROM dominios
WHERE variable = 'd11' AND TRY_CAST(valor AS INTEGER) BETWEEN 1 AND 16
""")
con.sql("SELECT * FROM map_motivo ORDER BY cod_motivo").show()

┌────────────┬─────────────────────────────────────────────────────────────────┬──────────────────────────┐
│ cod_motivo │                             motivo                              │       motivo_grupo       │
│   int32    │                             varchar                             │         varchar          │
├────────────┼─────────────────────────────────────────────────────────────────┼──────────────────────────┤
│          1 │ A trabajar                                                      │ Trabajo                  │
│          2 │ A estudiar                                                      │ Estudio                  │
│          3 │ A asuntos médicos personales                                    │ Salud o trámites         │
│          4 │ A visitar a alguien                                             │ Ocio, social o religioso │
│          5 │ A regresar al hogar                                             │ Regreso al hogar         │
│          6 │ A realizar al

**Resumen de ingeniería de características.**
- **Hogar:** estrato agrupado (bajo, medio, alto); ingreso en 4 rangos más "No reporta"; tenencia de vehículos resumida en `tiene_auto`, `tiene_moto` y `tiene_bici` (vacío = 0).
- **Persona:** rangos de edad; educación en 5 niveles; ocupación en 6 grupos; licencia de auto y de moto; `celular_datos` (tiene plan de datos).
- **Viaje:** motivo en 8 grupos; hora de salida (la primera del día viene de `d10a` y las siguientes de `d33` del viaje anterior); franja horaria; duración en minutos (válida entre 1 y 300); número de etapas.
- **Variable objetivo:** `es_individual` = 1 si el viaje es individual motorizado y 0 si es colectivo.

#### Decisiones – Ingeniería de características

| Variable nueva | Origen | Construcción | Justificación |
|---|---|---|---|
| `estrato_grupo` | `estrato` | Bajo (1-2), Medio (3-4), Alto (5-6) | Reduce categorías y facilita la segmentación |
| `ingreso_grupo` | `x1` | 4 rangos + "No reporta" | Agrupa 11 rangos con pocos registros en los extremos |
| `localidad` | `cod_loc` | Etiqueta según el diccionario | Interpretabilidad |
| `tiene_auto` | `c1a`, `c1b`, `c1c` | 1 si tiene al menos uno; vacío = 0 | Resume la tenencia de vehículo de 4 ruedas |
| `tiene_moto` | `c1e`, `c1f` | 1 si tiene al menos uno; vacío = 0 | Resume la tenencia de moto |
| `tiene_bici` | `c1m`, `c1n`, `c1o` | 1 si tiene al menos una; vacío = 0 | Resume la tenencia de bicicleta |
| `rango_edad` | `Age` | 05-17, 18-25, 26-35, 36-45, 46-60, 61+ | Etapas del ciclo de vida |
| `sexo` | `b5` | Etiqueta | Interpretabilidad |
| `nivel_educativo` | `b8` | 5 niveles | Agrupa 14 categorías |
| `ocupacion` | `b9a` | 6 grupos | Agrupa 24 categorías |
| `licencia_auto`, `licencia_moto` | `b19` | 1 si tiene la licencia | Capacidad de conducir |
| `celular_datos` | `b20` | 1 si tiene plan de datos | Requisito para usar apps de transporte |
| `motivo_grupo` | `d11` | 8 grupos | Agrupa 16 motivos |
| `salida_min` | `d10a`, `d33` | Primer viaje: `d10a`; siguientes: `d33` del viaje anterior | El formulario no registra la salida de cada viaje en una sola variable |
| `franja` | `salida_min` | Madrugada, pico AM, valle, pico PM, noche | Patrones de demanda por hora |
| `duracion_min` | `salida_min`, `d28` | Llegada − salida (+1440 si cruza medianoche); válida entre 1 y 300 | Medida del viaje; se excluyen valores atípicos |
| `num_etapas` | Tabla de etapas | Conteo de etapas por viaje | Complejidad del viaje |
| `es_individual` | Etapas (`d13`) | 1 = individual motorizado, 0 = colectivo | Variable objetivo |

Transformaciones

In [39]:
con.execute("""
CREATE OR REPLACE TABLE Dim_Hogar AS
SELECT ROW_NUMBER() OVER (ORDER BY key_hogar) AS sk_hogar, *
FROM fe_hogar;

CREATE OR REPLACE TABLE Dim_Persona AS
SELECT ROW_NUMBER() OVER (ORDER BY p.key_persona) AS sk_persona,
       h.sk_hogar,
       p.* EXCLUDE (key_hogar, fexp)
FROM fe_persona p
JOIN Dim_Hogar h USING (key_hogar);

CREATE OR REPLACE TABLE Dim_Motivo AS
SELECT ROW_NUMBER() OVER (ORDER BY cod_motivo) AS sk_motivo, *
FROM map_motivo;

CREATE OR REPLACE TABLE Dim_Modo AS
SELECT ROW_NUMBER() OVER (ORDER BY cod_modo) AS sk_modo, *
FROM map_modo
WHERE tipo_modo <> 'excluido';

CREATE OR REPLACE TABLE Dim_Franja AS
SELECT ROW_NUMBER() OVER (ORDER BY franja) AS sk_franja, franja
FROM (SELECT DISTINCT franja FROM fe_viaje);
""");

In [40]:
con.execute("""
CREATE OR REPLACE TABLE Fact_Viaje AS
SELECT ROW_NUMBER() OVER (ORDER BY v.key_viaje) AS sk_viaje,
       v.key_viaje,
       p.sk_persona,
       mo.sk_motivo,
       md.sk_modo,
       f.sk_franja,
       v.num_etapas,
       v.duracion_min,
       fp.fexp,
       v.tipo_transporte,
       v.es_individual
FROM fe_viaje v
JOIN Dim_Persona p  ON p.key_persona = v.key_persona
JOIN fe_persona  fp ON fp.key_persona = v.key_persona
JOIN Dim_Motivo  mo ON mo.cod_motivo = v.cod_motivo
JOIN Dim_Modo    md ON md.cod_modo   = v.cod_modo_principal
JOIN Dim_Franja  f  ON f.franja      = v.franja
""")

con.sql("""
SELECT (SELECT COUNT(*) FROM fe_viaje)   AS viajes_limpios,
       (SELECT COUNT(*) FROM Fact_Viaje) AS filas_fact,
       (SELECT COUNT(DISTINCT key_viaje) FROM Fact_Viaje) AS viajes_unicos_fact
""").show()

┌────────────────┬────────────┬────────────────────┐
│ viajes_limpios │ filas_fact │ viajes_unicos_fact │
│     int64      │   int64    │       int64        │
├────────────────┼────────────┼────────────────────┤
│          40529 │      40529 │              40529 │
└────────────────┴────────────┴────────────────────┘



In [41]:
con.sql("""
SELECT tipo_transporte,
       COUNT(*) AS viajes_muestra,
       ROUND(SUM(fexp)) AS viajes_expandidos,
       ROUND(100.0 * SUM(fexp) / SUM(SUM(fexp)) OVER (), 2) AS pct_ponderado
FROM Fact_Viaje GROUP BY ALL
""").show()

┌───────────────────────┬────────────────┬───────────────────┬───────────────┐
│    tipo_transporte    │ viajes_muestra │ viajes_expandidos │ pct_ponderado │
│        varchar        │     int64      │      double       │    double     │
├───────────────────────┼────────────────┼───────────────────┼───────────────┤
│ individual_motorizado │          17434 │         2939521.0 │         43.78 │
│ colectivo             │          23095 │         3774670.0 │         56.22 │
└───────────────────────┴────────────────┴───────────────────┴───────────────┘



In [42]:
base = con.sql("""
SELECT f.es_individual,
       f.fexp,
       h.estrato_grupo, h.ingreso_grupo, h.localidad,
       h.tiene_auto, h.tiene_moto, h.tiene_bici,
       p.edad, p.rango_edad, p.sexo, p.nivel_educativo, p.ocupacion,
       p.licencia_auto, p.licencia_moto, p.celular_datos,
       mo.motivo_grupo,
       fr.franja,
       f.num_etapas
FROM Fact_Viaje f
JOIN Dim_Persona p  USING (sk_persona)
JOIN Dim_Hogar   h  USING (sk_hogar)
JOIN Dim_Motivo  mo USING (sk_motivo)
JOIN Dim_Franja  fr USING (sk_franja)
""").df()

categoricas = ["estrato_grupo", "ingreso_grupo", "localidad", "rango_edad", "sexo",
               "nivel_educativo", "ocupacion", "motivo_grupo", "franja"]
base_modelo = pd.get_dummies(base, columns=categoricas, drop_first=True, dtype=int)
print(base.shape, "->", base_modelo.shape)
base_modelo.head()

(40529, 19) -> (40529, 62)


,es_individual,fexp,tiene_auto,tiene_moto,tiene_bici,edad,licencia_auto,licencia_moto,celular_datos,num_etapas,...,motivo_grupo_Estudio,"motivo_grupo_Ocio, social o religioso",motivo_grupo_Otro,motivo_grupo_Regreso al hogar,motivo_grupo_Salud o trámites,motivo_grupo_Trabajo,franja_2. Pico AM (5-9 h),franja_3. Valle (9-16 h),franja_4. Pico PM (16-20 h),franja_5. Noche (20-24 h)
0,1,274.883424,1,0,1,41,1,0,1,1,...,0,0,0,1,0,0,0,0,1,0
1,1,222.553295,0,0,0,61,0,0,1,1,...,0,0,0,0,0,1,0,1,0,0
2,0,69.779451,0,1,0,32,0,1,0,1,...,0,0,0,1,0,0,0,0,0,1
3,0,76.828315,0,1,0,26,0,0,0,1,...,0,0,0,1,0,0,0,1,0,0
4,1,107.800365,1,0,0,49,1,0,1,1,...,0,0,0,1,0,0,0,0,1,0


In [43]:
for t in ["Dim_Hogar", "Dim_Persona", "Dim_Motivo", "Dim_Modo", "Dim_Franja", "Fact_Viaje"]:
    con.execute(f"COPY {t} TO '{t}.parquet' (FORMAT parquet)")
con.execute("COPY (SELECT * FROM base_modelo) TO 'base_modelo.parquet' (FORMAT parquet)")

#### Decisiones – Transformaciones de datos

| Elemento | Decisión | Justificación |
|---|---|---|
| Esquema | Copo de nieve | `Dim_Hogar` depende de `Dim_Persona`, no directamente de la tabla de hechos |
| Grano de la tabla de hechos | Un viaje | El motivo y el modo son características del viaje |
| `Fact_Viaje` | Llaves a dimensiones, número de etapas, duración, `fexp` y variable objetivo | Medidas y atributos propios del viaje |
| `Dim_Hogar` | Localidad, estrato, ingreso y tenencia de vehículos | Atributos del hogar |
| `Dim_Persona` | Edad, sexo, educación, ocupación, licencias, celular y llave a `Dim_Hogar` | Atributos de la persona |
| `Dim_Motivo` | Motivo detallado y agrupado | Catálogo de motivos del diccionario |
| `Dim_Modo` | Modo detallado y tipo (colectivo / individual motorizado) | Catálogo de modos del diccionario |
| `Dim_Franja` | Franja horaria | Catálogo de franjas |
| Llaves sustitutas | `sk_` con `ROW_NUMBER()` | Independizar el modelo de las llaves de la encuesta |
| Validación | Filas de `Fact_Viaje` = viajes limpios | Confirmar que los JOINs no duplicaron registros |
| Ponderación | Conservar `fexp` en la tabla de hechos y en la base del modelo | Resultados representativos de Bogotá |
| Codificación | *One-hot encoding* con `drop_first` | Los modelos de clasificación requieren variables numéricas; `drop_first` evita la colinealidad perfecta |
| Exportación | Parquet | Conserva los tipos de datos y carga rápido |

**Resumen de transformaciones.**
- Se construyó un modelo **copo de nieve** con grano de un viaje: `Fact_Viaje` se conecta con `Dim_Persona`, `Dim_Motivo`, `Dim_Modo` y `Dim_Franja`, y `Dim_Persona` con `Dim_Hogar`. Cada dimensión tiene una llave sustituta (`sk_`).
- La tabla de hechos guarda número de etapas, duración, factor de expansión y la variable objetivo. Tiene [__] filas, igual al número de viajes limpios, así que los JOINs no duplicaron registros.
- Ponderado con `fexp`, el [__] % de los viajes es individual motorizado y el [__] % colectivo.
- Para el modelo se unió la tabla de hechos con sus dimensiones y las variables categóricas se codificaron con *one-hot encoding* (`drop_first`), lo que da [__] variables. `fexp` se conserva para usarlo como peso de las observaciones.
- Las tablas del modelo y la base analítica se exportaron en formato Parquet.